# M1 lab — how DistilBERT tokenizes and encodes text

DistilBERT is a BERT-family **encoder** for understanding tasks. The tokenization steps illustrated here also explain the token-to-ID idea used by generative models, although their tokenizers and architectures may differ.

Run in Google Colab or in a local Python 3.11+ environment. Install `m1/requirements.txt` first. The first run downloads the public `distilbert-base-uncased` tokenizer from Hugging Face; no model weights, GPU, account or dataset are needed.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers>=4.45,<5"])


## 1. Load the tokenizer

Use the tokenizer associated with the exact checkpoint. BERT and DistilBERT use WordPiece subword tokenization.

In [2]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
print(type(tokenizer).__name__, "vocabulary size:", tokenizer.vocab_size)

  from .autonotebook import tqdm as notebook_tqdm


DistilBertTokenizerFast vocabulary size: 30522


## 2. From text to tokens and IDs

`input_ids` are vocabulary indices, not semantic values. `[CLS]` and `[SEP]` delimit the input, and `##` marks a continuation subword.

In [3]:
text = "Tokenizing text is a core task of NLP."
encoded = tokenizer(text)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
for token, token_id in zip(tokens, encoded["input_ids"]):
    print(f"{token:>12}  {token_id}")
assert tokens[0] == "[CLS]" and tokens[-1] == "[SEP]"
assert len(tokens) == len(encoded["input_ids"])

       [CLS]  101
       token  19204
     ##izing  6026
        text  3793
          is  2003
           a  1037
        core  4563
        task  4708
          of  1997
          nl  17953
         ##p  2361
           .  1012
       [SEP]  102


## 3. Pad a batch and inspect the attention mask

The shorter sentence is padded to the length of the longer one. The mask has `1` for content and special tokens and `0` for padding.

In [4]:
texts = ["A short sentence.", "Tokenizing unfamiliar words is useful in NLP."]
batch = tokenizer(texts, padding=True, truncation=True)
for sentence, ids, mask in zip(texts, batch["input_ids"], batch["attention_mask"]):
    print("text:", sentence)
    print("tokens:", tokenizer.convert_ids_to_tokens(ids))
    print("input_ids:", ids)
    print("attention_mask:", mask)
    assert len(ids) == len(mask)
    assert all((token_id == tokenizer.pad_token_id) == (keep == 0) for token_id, keep in zip(ids, mask))

text: A short sentence.
tokens: ['[CLS]', 'a', 'short', 'sentence', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
input_ids: [101, 1037, 2460, 6251, 1012, 102, 0, 0, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
text: Tokenizing unfamiliar words is useful in NLP.
tokens: ['[CLS]', 'token', '##izing', 'unfamiliar', 'words', 'is', 'useful', 'in', 'nl', '##p', '.', '[SEP]']
input_ids: [101, 19204, 6026, 16261, 2616, 2003, 6179, 1999, 17953, 2361, 1012, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 4. Check the round trip

Decoding recovers readable text after removing the special tokens. Case and punctuation spacing may change because this checkpoint lowercases input.

In [5]:
recovered = tokenizer.decode(encoded["input_ids"], skip_special_tokens=True)
print("original:", text)
print("decoded: ", recovered)
assert "tokenizing" in recovered.lower()

original: Tokenizing text is a core task of NLP.
decoded:  tokenizing text is a core task of nlp.


## What to observe

1. Which words were split into WordPiece subwords?
2. What changes if you replace the first sentence with a much longer one?
3. Why does the padded position have an attention-mask value of zero?
